# Tests to plot functions and compare with matplotlib

In [ ]:
import time
import typing

import matplotlib.pyplot
import matplotlib.figure
import matplotlib.backends.backend_agg
import numpy
import scipy.special

from func_sketch._cpp import (
    ExpressionParser,
    PlotConfig,
    PlotRange,
    FunctionSampler,
    RGBColor,
    Plotter,
)

parser = ExpressionParser()
config = PlotConfig()
range = PlotRange((-3.0, 3.0), (-3.0, 3.0))
sampler = FunctionSampler(range, config)
line_color = RGBColor(0xCA, 0x76, 0x39)
plotter = Plotter(range, config)
height = 600
width = 800


def plot_function_in_func_sketch(
    expression_str: str, x_range: tuple[float, float], y_range: tuple[float, float]
) -> numpy.ndarray:
    """Plot a function using FuncSketch.

    Args:
        expression_str (str): Expression string of the function to plot.
        x_range (tuple[float, float]): Range of the x-axis.
        y_range (tuple[float, float]): Range of the y-axis.

    Returns:
        numpy.ndarray: Image of the plot.
    """
    image = numpy.ndarray((height, width, 3), dtype=numpy.uint8)

    start = time.perf_counter()
    range = PlotRange(x_range, y_range)
    sampler.range = range
    plotter.range = range
    expression = parser(expression_str)
    samples = sampler(expression)
    plotter.write_background(image)
    plotter.write_curve(samples, line_color, image)
    end = time.perf_counter()
    print(f"Plotting took {end - start} seconds")

    return image


def plot_function_in_matplotlib(
    function: typing.Callable[[numpy.ndarray], numpy.ndarray],
    x_range: tuple[float, float],
    y_range: tuple[float, float],
) -> numpy.ndarray:
    """Plot a function using Matplotlib.

    Args:
        function (typing.Callable[[numpy.ndarray], numpy.ndarray]): Function to plot.
        x_range (tuple[float, float]): Range of the x-axis.
        y_range (tuple[float, float]): Range of the y-axis.

    Returns:
        numpy.ndarray: Image of the plot.
    """
    start = time.perf_counter()
    x = numpy.linspace(x_range[0], x_range[1], width)
    y = function(x)
    dpi = 100
    figure = matplotlib.figure.Figure(figsize=(width / dpi, height / dpi), dpi=dpi)
    canvas = matplotlib.backends.backend_agg.FigureCanvasAgg(figure)
    axes = figure.gca()
    axes.set_xlim(x_range)
    axes.set_ylim(y_range)
    axes.plot(x, y, color=f"#{line_color.r:02X}{line_color.g:02X}{line_color.b:02X}")
    canvas.draw()
    image = numpy.asarray(canvas.buffer_rgba())
    end = time.perf_counter()
    print(f"Matplotlib plotting took {end - start} seconds")

    # Drop the alpha channel and convert to uint8
    image = image[:, :, :3].astype(numpy.uint8)

    return image


def plot_and_compare(
    expression_str: str,
    function: typing.Callable[[numpy.ndarray], numpy.ndarray],
    x_range: tuple[float, float],
    y_range: tuple[float, float],
) -> None:
    """Plot a function using both FuncSketch and Matplotlib, and compare the results.

    Args:
        expression_str (str): Expression string of the function to plot in FuncSketch.
        function (typing.Callable[[numpy.ndarray], numpy.ndarray]): Function to plot in Matplotlib.
        x_range (tuple[float, float]): Range of the x-axis.
        y_range (tuple[float, float]): Range of the y-axis.
    """
    func_sketch_image = plot_function_in_func_sketch(expression_str, x_range, y_range)
    matplotlib_image = plot_function_in_matplotlib(function, x_range, y_range)
    whole_image = numpy.concatenate((func_sketch_image, matplotlib_image), axis=1)
    matplotlib.pyplot.imshow(whole_image)
    matplotlib.pyplot.axis("off")
    matplotlib.pyplot.subplots_adjust(left=0, right=1, top=1, bottom=0)
    matplotlib.pyplot.show()

In [ ]:
plot_and_compare("x", lambda x: x, (-3.0, 3.0), (-3.0, 3.0))

In [ ]:
plot_and_compare("exp(x)", lambda x: numpy.exp(x), (-3.0, 3.0), (-1.0, 10.0))

In [ ]:
plot_and_compare(
    "gamma(x)", lambda x: scipy.special.gamma(x), (-3.0, 3.0), (-10.0, 10.0)
)